In [1]:
# ============================================
# CELL 1 — Import libraries and load the data
# ============================================

# pandas helps us work with CSV data like Excel
import pandas as pd

# numpy helps us do math calculations
import numpy as np

# os helps us work with files and folders
import os

# glob helps us find all CSV files in a folder
import glob

# Tell Python where our raw cricket data lives
DATA_FOLDER = "../data/raw/"

# Find ALL csv files inside that folder
# glob.glob finds every file matching a pattern
all_files = glob.glob(os.path.join(DATA_FOLDER, "*.csv"))

# Tell us how many match files we found
print(f"Total CSV files found: {len(all_files)}")

# Load the first file just to see what it looks like
sample = pd.read_csv(all_files[0])

# Show first 5 rows of sample file
print("\nSample file columns:")
print(sample.columns.tolist())

# Show shape — how many rows and columns
print(f"\nShape: {sample.shape}")

# Show first 3 rows
print("\nFirst 3 rows:")
sample.head(3)

Total CSV files found: 2478

Sample file columns:
['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed']

Shape: (100, 22)

First 3 rows:


,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,...,extras,wides,noballs,byes,legbyes,penalty,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed
0,598068,2013,2013-05-18,M Chinnaswamy Stadium,1,0.1,Royal Challengers Bangalore,Chennai Super Kings,V Kohli,CH Gayle,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,598068,2013,2013-05-18,M Chinnaswamy Stadium,1,0.2,Royal Challengers Bangalore,Chennai Super Kings,V Kohli,CH Gayle,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,598068,2013,2013-05-18,M Chinnaswamy Stadium,1,0.3,Royal Challengers Bangalore,Chennai Super Kings,V Kohli,CH Gayle,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# ============================================
# CELL 2 — Load ALL 2478 CSV files into one big dataframe
# ============================================

# Empty list to store each match's data
all_matches = []

# Loop through every single CSV file we found
for file in all_files:
    # Read each CSV file into a dataframe
    df = pd.read_csv(file)
    # Add it to our list
    all_matches.append(df)

# Combine ALL matches into one giant dataframe
# ignore_index=True means reset row numbers from 0
data = pd.concat(all_matches, ignore_index=True)

# Tell us the total size of our dataset
print(f"Total deliveries loaded: {len(data)}")
print(f"Total columns: {len(data.columns)}")
print(f"Seasons available: {sorted(data['season'].unique())}")
print(f"Total unique matches: {data['match_id'].nunique()}")

ParserError: Error tokenizing data. C error: Expected 3 fields in line 22, saw 4


In [3]:
# ============================================
# CELL 2 — Load ALL CSV files, skip bad ones
# ============================================

# Empty list to store each match's data
all_matches = []

# Count how many files we skip
skipped = 0

# Loop through every single CSV file we found
for file in all_files:
    try:
        # on_bad_lines='skip' tells pandas to ignore broken rows
        df = pd.read_csv(file, on_bad_lines='skip')
        
        # Only keep files that have match data columns
        if 'innings' in df.columns and 'ball' in df.columns:
            all_matches.append(df)
        else:
            skipped += 1
            
    except Exception as e:
        # If file still errors just skip it completely
        skipped += 1

# Combine ALL good matches into one giant dataframe
data = pd.concat(all_matches, ignore_index=True)

print(f"Files skipped: {skipped}")
print(f"Total deliveries loaded: {len(data)}")
print(f"Total unique matches: {data['match_id'].nunique()}")
print(f"Seasons available: {sorted(data['season'].unique())}")


Files skipped: 1239
Total deliveries loaded: 294757
Total unique matches: 1239


TypeError: '<' not supported between instances of 'str' and 'int'

In [4]:
# ============================================
# CELL 3 — Clean the data and engineer features
# ============================================

# Fill empty/NaN values with 0 for number columns
# NaN means "empty cell" in pandas
data['runs_off_bat'] = data['runs_off_bat'].fillna(0)
data['extras'] = data['extras'].fillna(0)
data['wides'] = data['wides'].fillna(0)
data['noballs'] = data['noballs'].fillna(0)

# Create a new column — total runs on that delivery
# runs_off_bat + extras = total runs scored that ball
data['total_runs'] = data['runs_off_bat'] + data['extras']

# Create a wicket column — 1 if wicket fell, 0 if not
# If wicket_type is not empty, a wicket fell
data['is_wicket'] = data['wicket_type'].notna().astype(int)

# Show what we have now
print("Columns we have:")
print(data.columns.tolist())
print(f"\nTotal wickets in dataset: {data['is_wicket'].sum()}")
print(f"\nSample of total_runs column:")
print(data['total_runs'].value_counts().head())

Columns we have:
['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'total_runs', 'is_wicket']

Total wickets in dataset: 14650

Sample of total_runs column:
total_runs
1    122606
0    100919
4     34690
2     19255
6     15607
Name: count, dtype: int64


In [5]:
# ============================================
# CELL 4 — Build match-level cumulative features
# ============================================

# We need to calculate for each ball:
# - runs scored SO FAR in the innings
# - wickets fallen SO FAR
# - balls bowled SO FAR

# Sort data by match, innings, and ball number
# This makes sure everything is in the right order
data = data.sort_values(['match_id', 'innings', 'ball']).reset_index(drop=True)

# For each match+innings, calculate cumulative runs so far
# cumsum() adds up values row by row
data['cum_runs'] = data.groupby(['match_id', 'innings'])['total_runs'].cumsum()

# For each match+innings, calculate cumulative wickets so far
data['cum_wickets'] = data.groupby(['match_id', 'innings'])['is_wicket'].cumsum()

# Calculate ball number (1st ball, 2nd ball etc) within each innings
data['ball_number'] = data.groupby(['match_id', 'innings']).cumcount() + 1

# Calculate balls remaining (T20 = 120 balls total)
data['balls_remaining'] = 120 - data['ball_number']

# Show sample
print("New features created!")
print(data[['match_id', 'innings', 'ball', 'total_runs', 
            'cum_runs', 'cum_wickets', 'ball_number', 
            'balls_remaining']].head(10))

New features created!
   match_id  innings  ball  total_runs  cum_runs  cum_wickets  ball_number  \
0    335982        1   0.1           1         1            0            1   
1    335982        1   0.2           0         1            0            2   
2    335982        1   0.3           1         2            0            3   
3    335982        1   0.4           0         2            0            4   
4    335982        1   0.5           0         2            0            5   
5    335982        1   0.6           0         2            0            6   
6    335982        1   0.7           1         3            0            7   
7    335982        1   1.1           0         3            0            8   
8    335982        1   1.2           4         7            0            9   
9    335982        1   1.3           4        11            0           10   

   balls_remaining  
0              119  
1              118  
2              117  
3              116  
4             

In [6]:
# ============================================
# CELL 5 — Calculate the TARGET score and WIN LABEL
# ============================================

# For each match, find the total runs scored in innings 1
# This becomes the target for innings 2
innings1 = data[data['innings'] == 1].groupby('match_id')['total_runs'].sum().reset_index()
innings1.columns = ['match_id', 'target']

# Add 1 to target because team 2 needs to BEAT the score
innings1['target'] = innings1['target'] + 1

# Merge target back into main data
data = data.merge(innings1, on='match_id', how='left')

# For innings 2 — calculate runs still needed to win
data['runs_needed'] = data['target'] - data['cum_runs']

# Make sure runs_needed is never negative
data['runs_needed'] = data['runs_needed'].clip(lower=0)

# Now figure out who WON each match
# Find the final score of innings 2 for each match
innings2_final = data[data['innings'] == 2].groupby('match_id').agg(
    final_runs=('cum_runs', 'max'),      # highest cumulative runs = final score
    final_wickets=('cum_wickets', 'max') # highest cumulative wickets
).reset_index()

# Team 2 won if their final runs >= target
innings2_final['team2_won'] = (innings2_final['final_runs'] >= innings2_final['target'] - 1).astype(int)

# Merge win result back into main data
data = data.merge(innings2_final[['match_id', 'team2_won']], on='match_id', how='left')

print(f"Matches where team2 won: {innings2_final['team2_won'].sum()}")
print(f"Matches where team1 won: {(innings2_final['team2_won'] == 0).sum()}")
print(f"\nSample of target column:")
print(data[['match_id', 'innings', 'cum_runs', 'target', 'runs_needed']].head(10))

KeyError: 'target'

In [7]:
# ============================================
# CELL 5 — Calculate TARGET score and WIN LABEL
# ============================================

# For each match, find the total runs scored in innings 1
# This becomes the target for innings 2
innings1 = data[data['innings'] == 1].groupby('match_id')['total_runs'].sum().reset_index()
innings1.columns = ['match_id', 'target']

# Add 1 to target because team 2 needs to BEAT the score
innings1['target'] = innings1['target'] + 1

# Merge target back into main data FIRST
data = data.merge(innings1, on='match_id', how='left')

# For innings 2 — calculate runs still needed to win
data['runs_needed'] = data['target'] - data['cum_runs']

# Make sure runs_needed is never negative
data['runs_needed'] = data['runs_needed'].clip(lower=0)

# Now figure out who WON each match
# Find the final score of innings 2 for each match
innings2_final = data[data['innings'] == 2].groupby('match_id').agg(
    final_runs=('cum_runs', 'max'),
    final_wickets=('cum_wickets', 'max'),
    target=('target', 'max')        # ← get target here too
).reset_index()

# Team 2 won if their final runs >= target
innings2_final['team2_won'] = (
    innings2_final['final_runs'] >= innings2_final['target'] - 1
).astype(int)

# Merge win result back into main data
data = data.merge(innings2_final[['match_id', 'team2_won']], on='match_id', how='left')

print(f"Matches where team2 won: {innings2_final['team2_won'].sum()}")
print(f"Matches where team1 won: {(innings2_final['team2_won']==0).sum()}")
print(f"\nSample:")
print(data[['match_id','innings','cum_runs','target','runs_needed']].head(10))

KeyError: 'target'

In [8]:
# ============================================
# QUICK FIX — Remove duplicate columns if they exist
# ============================================

# Check what columns we currently have
print("Current columns:")
print(data.columns.tolist())

# Drop target related columns if they already exist
# This prevents duplicate column errors
cols_to_drop = ['target', 'runs_needed', 'team2_won']
for col in cols_to_drop:
    if col in data.columns:
        data = data.drop(columns=[col])
        print(f"Dropped: {col}")

print("\nColumns after cleanup:")
print(data.columns.tolist())

Current columns:
['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'total_runs', 'is_wicket', 'cum_runs', 'cum_wickets', 'ball_number', 'balls_remaining', 'target_x', 'runs_needed', 'target_y']
Dropped: runs_needed

Columns after cleanup:
['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'total_runs', 'is_wicket', 'cum_runs', 'cum_wickets', 'ball_number', 'balls_remaining', 'target_x', 'target_y']


In [9]:
# ============================================
# CLEAN FIX — Remove duplicate target columns
# ============================================

# Drop both duplicate target columns
data = data.drop(columns=['target_x', 'target_y'])

# Now calculate target FRESH and correctly
# Get total runs scored in innings 1 for each match
innings1 = data[data['innings'] == 1].groupby(
    'match_id')['total_runs'].sum().reset_index()

# Rename column to target
innings1.columns = ['match_id', 'target']

# Team 2 needs 1 more run than team 1 scored
innings1['target'] = innings1['target'] + 1

# Merge target into data ONE time only
data = data.merge(innings1, on='match_id', how='left')

# Calculate runs needed for innings 2
data['runs_needed'] = (data['target'] - data['cum_runs']).clip(lower=0)

# Find who won each match
innings2_final = data[data['innings'] == 2].groupby('match_id').agg(
    final_runs=('cum_runs', 'max'),
    target=('target', 'max')
).reset_index()

# Team 2 won if they reached the target
innings2_final['team2_won'] = (
    innings2_final['final_runs'] >= innings2_final['target'] - 1
).astype(int)

# Merge win result back
data = data.merge(
    innings2_final[['match_id', 'team2_won']], 
    on='match_id', 
    how='left'
)

print(f"✅ Team 2 won: {innings2_final['team2_won'].sum()} matches")
print(f"✅ Team 1 won: {(innings2_final['team2_won']==0).sum()} matches")
print(f"✅ Target column sample: {data['target'].head(3).tolist()}")
print(f"✅ Columns now: {data.columns.tolist()}")

✅ Team 2 won: 666 matches
✅ Team 1 won: 567 matches
✅ Target column sample: [223, 223, 223]
✅ Columns now: ['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'total_runs', 'is_wicket', 'cum_runs', 'cum_wickets', 'ball_number', 'balls_remaining', 'target', 'runs_needed', 'team2_won']


In [10]:
# ============================================
# CELL 6 — Build the ML training dataset
# ============================================

# We only want innings 2 data for win probability
# Because that's when we know the target and can predict winner
innings2_data = data[data['innings'] == 2].copy()

# These are the FEATURES our model will use to predict
# Think of these as "clues" we give the model
features = [
    'cum_runs',        # runs scored so far in innings 2
    'cum_wickets',     # wickets lost so far
    'balls_remaining', # balls left to play
    'runs_needed',     # runs still needed to win
    'ball_number',     # which ball we are on
    'target',          # what score team 2 is chasing
]

# This is what we want to PREDICT — did team 2 win?
target_col = 'team2_won'

# Create X (features) and y (answer)
X = innings2_data[features]
y = innings2_data[target_col]

# Drop any rows where data is missing
# dropna removes rows with empty values
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

print(f"✅ Training samples: {len(X)}")
print(f"✅ Features: {features}")
print(f"✅ Win rate in dataset: {y.mean():.2%}")
print(f"\nSample of X:")
print(X.head())

✅ Training samples: 141823
✅ Features: ['cum_runs', 'cum_wickets', 'balls_remaining', 'runs_needed', 'ball_number', 'target']
✅ Win rate in dataset: 52.64%

Sample of X:
     cum_runs  cum_wickets  balls_remaining  runs_needed  ball_number  target
124         1            0              119          222            1     223
125         2            0              118          221            2     223
126         2            0              117          221            3     223
127         3            0              116          220            4     223
128         4            0              115          219            5     223


In [12]:
# ============================================
# CELL 7 — Train the XGBoost model!
# ============================================

# Import train_test_split to split data into training and testing
from sklearn.model_selection import train_test_split

# Import XGBoost — our main ML model
from xgboost import XGBClassifier

# Import accuracy score to measure how good our model is
from sklearn.metrics import accuracy_score, classification_report

# Split data — 80% for training, 20% for testing
# random_state=42 means we get same split every time
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% goes to testing
    random_state=42     # same split every time
)

print(f"✅ Training samples: {len(X_train)}")
print(f"✅ Testing samples: {len(X_test)}")

# Create the XGBoost model
# n_estimators = number of trees to build
# max_depth = how deep each tree goes
# learning_rate = how fast model learns
model = XGBClassifier(
    n_estimators=100,    # build 100 trees
    max_depth=6,         # each tree 6 levels deep
    learning_rate=0.1,   # learning speed
    random_state=42,     # reproducible results
    eval_metric='logloss' # how we measure error
)

# TRAIN THE MODEL! This is the magic moment! 🎉
print("\n⏳ Training model... please wait...")
model.fit(X_train, y_train)

# Test how accurate our model is
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n🎉 Model Accuracy: {accuracy:.2%}")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred))

✅ Training samples: 113458
✅ Testing samples: 28365

⏳ Training model... please wait...

🎉 Model Accuracy: 81.19%

Detailed Report:
              precision    recall  f1-score   support

         0.0       0.82      0.78      0.80     13516
         1.0       0.81      0.84      0.82     14849

    accuracy                           0.81     28365
   macro avg       0.81      0.81      0.81     28365
weighted avg       0.81      0.81      0.81     28365



In [13]:
# ============================================
# CELL 8 — Add SHAP values (the "WHY" explainer)
# ============================================

# Import SHAP — explains WHY model made each prediction
import shap

print("⏳ Calculating SHAP values... please wait...")

# Create a SHAP explainer using our trained model
# TreeExplainer works specifically with XGBoost
explainer = shap.TreeExplainer(model)

# Calculate SHAP values for test data
# We use first 500 rows only — faster to calculate
shap_values = explainer.shap_values(X_test[:500])

# Show a sample explanation for ONE delivery
# Let's look at ball number 50 as example
sample_idx = 50
sample_ball = X_test.iloc[sample_idx]
sample_shap = shap_values[sample_idx]

print(f"\n✅ SHAP values calculated!")
print(f"\nSample ball situation:")
print(f"  Runs scored so far: {sample_ball['cum_runs']}")
print(f"  Wickets lost: {sample_ball['cum_wickets']}")
print(f"  Balls remaining: {sample_ball['balls_remaining']}")
print(f"  Runs needed: {sample_ball['runs_needed']}")

print(f"\nSHAP impact on win probability:")
for feat, shap_val in zip(features, sample_shap):
    direction = "↑ helps batting team" if shap_val > 0 else "↓ hurts batting team"
    print(f"  {feat}: {shap_val:.4f} {direction}")

# Get win probability for this ball
win_prob = model.predict_proba(X_test[:500])[sample_idx][1]
print(f"\n🎯 Win probability at this moment: {win_prob:.2%}")

⏳ Calculating SHAP values... please wait...

✅ SHAP values calculated!

Sample ball situation:
  Runs scored so far: 18
  Wickets lost: 0
  Balls remaining: 102
  Runs needed: 75

SHAP impact on win probability:
  cum_runs: -0.1008 ↓ hurts batting team
  cum_wickets: 0.8699 ↑ helps batting team
  balls_remaining: 0.4536 ↑ helps batting team
  runs_needed: 0.3552 ↑ helps batting team
  ball_number: 0.0000 ↓ hurts batting team
  target: 3.6155 ↑ helps batting team

🎯 Win probability at this moment: 99.50%


In [14]:
# ============================================
# CELL 9 — Build the PRESSURE SCORE formula
# This is YOUR original invention for the project!
# ============================================

def calculate_pressure_score(runs_needed, balls_remaining, 
                               wickets_fallen, win_probability):
    """
    Pressure Score = custom metric invented for CricIQ
    
    Higher score = MORE pressure on batting team
    Scale: 0 to 100
    
    Formula factors:
    - Required Run Rate vs balls remaining
    - Wickets in hand (10 - wickets fallen)
    - Win probability (low win prob = high pressure)
    """
    
    # Avoid division by zero
    # If no balls remaining, maximum pressure
    if balls_remaining <= 0:
        return 100.0
    
    # Calculate required run rate per ball
    # runs_needed divided by balls_remaining
    required_rate = runs_needed / balls_remaining
    
    # Wickets in hand — more wickets = less pressure
    wickets_in_hand = 10 - wickets_fallen
    
    # Wicket pressure — fewer wickets = more pressure
    # We normalize to 0-1 scale
    wicket_pressure = 1 - (wickets_in_hand / 10)
    
    # Run rate pressure — higher required rate = more pressure
    # We cap it at 3 runs per ball (extreme pressure)
    run_rate_pressure = min(required_rate / 3, 1.0)
    
    # Win probability pressure — lower win prob = more pressure
    win_pressure = 1 - win_probability
    
    # FINAL PRESSURE SCORE
    # Weighted combination of all three factors
    # win_pressure gets highest weight (50%)
    # run_rate_pressure gets 30%
    # wicket_pressure gets 20%
    pressure = (
        (win_pressure * 0.50) +      # 50% weight
        (run_rate_pressure * 0.30) + # 30% weight
        (wicket_pressure * 0.20)     # 20% weight
    ) * 100  # multiply by 100 to get 0-100 scale
    
    return round(pressure, 2)

# Test our pressure score with some scenarios
print("🔥 PRESSURE SCORE EXAMPLES:")
print("="*50)

# Scenario 1 — Easy chase, early overs
p1 = calculate_pressure_score(
    runs_needed=150, balls_remaining=100, 
    wickets_fallen=0, win_probability=0.3)
print(f"Need 150 from 100 balls, 0 wickets lost: {p1}/100")

# Scenario 2 — Tight finish
p2 = calculate_pressure_score(
    runs_needed=30, balls_remaining=18, 
    wickets_fallen=7, win_probability=0.45)
print(f"Need 30 from 18 balls, 7 wickets lost: {p2}/100")

# Scenario 3 — Almost impossible
p3 = calculate_pressure_score(
    runs_needed=60, balls_remaining=12, 
    wickets_fallen=8, win_probability=0.05)
print(f"Need 60 from 12 balls, 8 wickets lost: {p3}/100")

# Scenario 4 — Comfortable chase
p4 = calculate_pressure_score(
    runs_needed=20, balls_remaining=40, 
    wickets_fallen=2, win_probability=0.85)
print(f"Need 20 from 40 balls, 2 wickets lost: {p4}/100")

🔥 PRESSURE SCORE EXAMPLES:
Need 150 from 100 balls, 0 wickets lost: 50.0/100
Need 30 from 18 balls, 7 wickets lost: 58.17/100
Need 60 from 12 balls, 8 wickets lost: 93.5/100
Need 20 from 40 balls, 2 wickets lost: 16.5/100


In [15]:
# ============================================
# CELL 10 — Save the model to disk
# We save it so our FastAPI backend can load it
# ============================================

# joblib helps us save and load ML models
import joblib

# os helps us create folders
import os

# Create the ml folder inside backend if it doesn't exist
os.makedirs('../backend/ml', exist_ok=True)

# Save the XGBoost model to a file
# This file will be loaded by our FastAPI backend later
joblib.dump(model, '../backend/ml/win_probability_model.pkl')
print("✅ Model saved to backend/ml/win_probability_model.pkl")

# Save the SHAP explainer too
joblib.dump(explainer, '../backend/ml/shap_explainer.pkl')
print("✅ SHAP explainer saved to backend/ml/shap_explainer.pkl")

# Save the feature names so backend knows what order to use
joblib.dump(features, '../backend/ml/feature_names.pkl')
print("✅ Feature names saved to backend/ml/feature_names.pkl")

# Save the pressure score function as a separate file
# We write it directly to a Python file
pressure_code = '''
# pressure_score.py
# Custom Pressure Score metric invented for CricIQ

def calculate_pressure_score(runs_needed, balls_remaining, 
                               wickets_fallen, win_probability):
    if balls_remaining <= 0:
        return 100.0
    required_rate = runs_needed / balls_remaining
    wickets_in_hand = 10 - wickets_fallen
    wicket_pressure = 1 - (wickets_in_hand / 10)
    run_rate_pressure = min(required_rate / 3, 1.0)
    win_pressure = 1 - win_probability
    pressure = (
        (win_pressure * 0.50) +
        (run_rate_pressure * 0.30) +
        (wicket_pressure * 0.20)
    ) * 100
    return round(pressure, 2)
'''

# Write pressure score to a Python file
with open('../backend/ml/pressure_score.py', 'w') as f:
    f.write(pressure_code)
print("✅ Pressure score saved to backend/ml/pressure_score.py")

print("\n🎉 ALL ML FILES SAVED SUCCESSFULLY!")
print("\nFiles created:")
print("  backend/ml/win_probability_model.pkl")
print("  backend/ml/shap_explainer.pkl")
print("  backend/ml/feature_names.pkl")
print("  backend/ml/pressure_score.py")

✅ Model saved to backend/ml/win_probability_model.pkl
✅ SHAP explainer saved to backend/ml/shap_explainer.pkl
✅ Feature names saved to backend/ml/feature_names.pkl
✅ Pressure score saved to backend/ml/pressure_score.py

🎉 ALL ML FILES SAVED SUCCESSFULLY!

Files created:
  backend/ml/win_probability_model.pkl
  backend/ml/shap_explainer.pkl
  backend/ml/feature_names.pkl
  backend/ml/pressure_score.py
